# Why `deprotonate` comes before `attach`

`attach` replaces a hydrogen with a bond, so the anchor atom keeps its formal
charge. That is right for a reaction whose product keeps the charge, and wrong
for one whose product does not. Acylation is the second kind: a lysine
side-chain amine is protonated at pH 7, but only the neutral amine reacts, and
the product is a neutral amide. An acetyl group is attached to lysine 48 of
ubiquitin with and without deprotonating the amine first, and the product is
compared with the CCD's own N6-acetyl-lysine, `ALY`.

In [1]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [2]:
from mbuild.biopolymers import CCDLibrary, Protein, fragment_from_smiles
from openff.toolkit import Molecule

mbuild_acetyl = fragment_from_smiles("*C(=O)C", "ACE")
# The CCD describes the product: N6-acetyl-L-lysine, component ALY.
aly = CCDLibrary(download=True)["ALY"][0]
aly_nz_charge = aly.name_to_atom["NZ"].formal_charge
print(f"CCD ALY: NZ formal charge {aly_nz_charge:+d}, residue formal charge {aly.formal_charge:+d}")


def report(label, mbuild_protein, product=False):
    residue = mbuild_protein.get_residue(48)
    openff_molecule = Molecule.from_rdkit(mbuild_protein.to_rdkit(), allow_undefined_stereo=True)
    verdict = ""
    if product:
        verdict = "matches ALY" if residue.atom_formal_charges.get("NZ", 0) == aly_nz_charge else "amide nitrogen keeps +1: wrong"
    print(
        f"{label:26s} LYS48 {residue.formal_charge:+d}   protein net {mbuild_protein.net_formal_charge:+d}   "
        f"exported {openff_molecule.total_charge}   {verdict}"
    )

In [4]:
mbuild_protein = Protein("../1ubq_protonated.pdb")
report("as loaded", mbuild_protein)

mbuild_protein.attach(mbuild_acetyl, resnum=48, atom_name="NZ", relax=False)
report("attach only", mbuild_protein, product=True)

as loaded                    LYS48 +1   net +0   exported 0.0 elementary_charge
attach only                  LYS48 +1   net +0   exported 0.0 elementary_charge


In [5]:
mbuild_protein = Protein("../1ubq_protonated.pdb")
mbuild_protein.deprotonate(48, "NZ")
report("deprotonate", mbuild_protein)

mbuild_protein.attach(mbuild_acetyl, resnum=48, atom_name="NZ", relax=False)
report("deprotonate, then attach", mbuild_protein, product=True)
assert mbuild_protein.get_residue(48).atom_formal_charges.get("NZ", 0) == aly_nz_charge

deprotonate                  LYS48 +0   net -1   exported -1.0 elementary_charge
deprotonate, then attach     LYS48 +0   net -1   exported -1.0 elementary_charge


`attach` never changes a formal charge, so the site has to be in the form that
reacts before the bond forms. For an acylation that is the neutral amine, and
`deprotonate` is the step that gets it there. The rule is the same for every
site that reacts from its neutral or anionic form: the hydroxyls of serine,
threonine and tyrosine, and the thiol of cysteine.